#### Import required libraries

In [1]:
import os
import glob
import warnings
import numpy as np
import pandas as pd
import tkinter as tk
import tifffile as tf
from tkinter import filedialog
warnings.filterwarnings("ignore")

#### Define custom functions

In [2]:
def fileWalker(file_extension, strfilter = False, filter_string = ""):
    window = tk.Tk()
    window.withdraw()
    path = filedialog.askdirectory()
    files = []

    # r=root, d=directories, f = files
    for r, d, f in os.walk(path):
        for file in f:
            if file_extension in file:
                if strfilter:
                    if filter_string in file:
                        print(os.path.join(r, file))
                        files.append(os.path.join(r, file))
                else:
                    print(os.path.join(r, file))
                    files.append(os.path.join(r, file))
    window.destroy()
    return files

#### Get list of single channel motion corrected data files

In [3]:
files = fileWalker(".tif",strfilter=True,filter_string="motion_corrected")

/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B32/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B32/Motion_corrected_tiff_files/R1_plane1_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B35/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B34/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B33/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B42/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B38/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B36/Motion_corrected_tiff_files/R1_plane0_motion_corrected_mas

In [4]:
metadata_dict = pd.read_pickle("../metadata/brain_metadata_dict.pkl")
metadata_dict

{'AKG_B30': {'R1': {'fps': np.float64(1.254226807851382),
   'timestamps': array([3.13881300e+02, 7.02414300e+02, 1.04508230e+03, ...,
          2.09928224e+06, 2.09970629e+06, 2.10009863e+06], shape=(5268,))}},
 'AKG_B31': {'R1': {'fps': np.float64(1.3772295654583815),
   'timestamps': array([1.91956500e+02, 5.10254500e+02, 8.52181500e+02, ...,
          2.09983864e+06, 2.10022327e+06, 2.10059388e+06], shape=(5786,))}},
 'AKG_B32': {'R1': {'fps': np.float64(1.2513626234265482),
   'timestamps': array([1.90762000e+02, 5.46099000e+02, 8.96003000e+02, ...,
          2.09933303e+06, 2.09971717e+06, 2.10011067e+06], shape=(5256,))}},
 'AKG_B33': {'R1': {'fps': np.float64(1.281480949640285),
   'timestamps': array([3.05935450e+02, 6.52920450e+02, 1.03718245e+03, ...,
          2.09991790e+06, 2.10031088e+06, 2.10069451e+06], shape=(5384,))}},
 'AKG_B34': {'R1': {'fps': np.float64(1.3167183379336684),
   'timestamps': array([1.97150700e+02, 5.75101700e+02, 9.04908700e+02, ...,
          2.09

In [5]:
%%time

for f in files:
    
    print(f)
    
    raw_data = tf.imread(f)
    
    filedir = os.path.dirname(f).split("/Motion")[0]
    
    plane_id = os.path.basename(f).split(".tif")[0].split("_")[1]
    
    masks_temp = glob.glob(f'{filedir}/Masks/*Mask*')
    masks = []
    for mt in masks_temp:
        if plane_id in mt:
            masks.append(mt)
    
    brain_id = f"AKG_{f.split('AKG_')[1].split('/')[0]}"
    recording_id = os.path.basename(f).split(".tif")[0].split("_")[0]
    
    if plane_id == "plane0":
        timestamps = metadata_dict[brain_id][recording_id]["timestamps"][::2]
    elif plane_id == "plane1":
        timestamps = metadata_dict[brain_id][recording_id]["timestamps"][1::2]
    
    df = pd.DataFrame()
    df["absolute_timestamps(ms)"] = timestamps
    df["relative_timestamps(ms)"] = timestamps-timestamps[0]
    
    for m in masks:
        mask_image = tf.imread(m)
        mask_array = (mask_image>0).astype(int)
        mask = pd.DataFrame(mask_array).replace(0,np.nan).values
        mask_id = os.path.basename(m).split(".tif")[0]
        masked_data = np.multiply(raw_data,mask)
        masked_data_trace = np.nanmean(np.nanmean(masked_data,axis=1),axis=1)
        df[f"F_trace_{mask_id}"] = masked_data_trace
    
    df.to_csv(f"{filedir}/csv_files/{recording_id}_{plane_id}_data.csv",index=None)

/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B32/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B32/Motion_corrected_tiff_files/R1_plane1_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B35/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B34/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B33/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B42/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B38/Motion_corrected_tiff_files/R1_plane0_motion_corrected_masked.tif
/Users/cosmos/Desktop/RashmiExperiments/Github/data/AKG_B36/Motion_corrected_tiff_files/R1_plane0_motion_corrected_mas